In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

class GapFillOrBounceStrategy:
    def __init__(self, data, close_column_name):
        """
        Initializes the strategy with stock data.
        
        Parameters:
        data (pd.DataFrame): A DataFrame containing stock data with 'Open', 'Close', 'High', 'Low', 'Adj Close' columns.
        close_column_name (str): The name of the column to use for gap calculations.
        """
        self.data = data
        self.close_column_name = close_column_name

    def identify_gaps(self, gap_percentage=1.0, window_size=1):
        """
        Identifies gaps in the stock data based on the specified gap percentage and window size.
        
        Parameters:
        gap_percentage (float): The minimum gap size as a percentage to consider for the strategy.
        window_size (int): The number of samples to consider for identifying a gap.
        
        Returns:
        pd.DataFrame: A DataFrame with gap information.
        """
        self.data['PrevClose'] = self.data[self.close_column_name].shift(window_size)
        self.data['Gap'] = self.data['Open'] - self.data['PrevClose']
        self.data['GapPercent'] = (self.data['Gap'] / self.data['PrevClose']) * 100
        self.data['GapDirection'] = np.where(self.data['Gap'] > 0, 'Up', 'Down')
        self.data['IsGap'] = np.where(np.abs(self.data['GapPercent']) >= gap_percentage, True, False)
        
        self.data['GapTop'] = np.where(self.data['IsGap'], self.data['PrevClose'], np.nan)
        self.data['GapBottom'] = np.where(self.data['IsGap'], self.data['Open'], np.nan)
        self.data['GapBottom'] = self.data['GapBottom'].ffill()
        
        return self.data

    def generate_signals(self, long_entry=True, short_entry=True):
        """
        Generates entry and exit signals based on the gap fill or bounce strategy.
        
        Parameters:
        long_entry (bool): Whether to include long entry signals.
        short_entry (bool): Whether to include short entry signals.
        
        Returns:
        pd.DataFrame: A DataFrame with entry and exit signals.
        """
        self.long_entry = long_entry
        self.short_entry = short_entry
        
        self.data['LongEntry'] = np.nan
        self.data['LongExit'] = np.nan
        self.data['ShortEntry'] = np.nan
        self.data['ShortExit'] = np.nan
        
        if self.long_entry:
            self.data['LongEntry'] = np.where((self.data['IsGap']) & (self.data['GapDirection'] == 'Down') & (self.data[self.close_column_name] > self.data['Open']), self.data['Open'], np.nan)
            self.data['LongExit'] = np.where((self.data['LongEntry'].notna()) & (self.data[self.close_column_name] >= self.data['GapTop']), self.data['GapTop'], np.nan)
        
        if self.short_entry:
            self.data['ShortEntry'] = np.where((self.data['IsGap']) & (self.data['GapDirection'] == 'Up') & (self.data[self.close_column_name] < self.data['Open']), self.data['Open'], np.nan)
            self.data['ShortExit'] = np.where((self.data['ShortEntry'].notna()) & (self.data[self.close_column_name] <= self.data['GapBottom']), self.data['GapBottom'], np.nan)
        
        return self.data

    def backtest(self, investment_amount=10000):
        """
        Backtests the strategy on the provided data using the specified column for profit/loss calculations.
        
        Parameters:
        investment_amount (float): The amount of money to invest in each trade.
        
        Returns:
        pd.DataFrame: A DataFrame with the backtesting results.
        """
        self.data['Position'] = 0
        if self.long_entry:
            self.data['Position'] = np.where(self.data['LongEntry'].notna(), 1, self.data['Position'])
        if self.short_entry:
            self.data['Position'] = np.where(self.data['ShortEntry'].notna(), -1, self.data['Position'])

        self.data['PnLPercent'] = np.where(self.data['Position'] == 1,
                                           (self.data[self.close_column_name] - self.data['Open']) / self.data['Open'],
                                           np.where(self.data['Position'] == -1,
                                                    (self.data['Open'] - self.data[self.close_column_name]) / self.data['Open'],
                                                    0))
        
        self.data['PnL'] = self.data['PnLPercent'] * investment_amount
        self.data['CumulativePnL'] = self.data['PnL'].cumsum()
        
        return self.data

# Example usage
# Download last month of IJR data with 30 minutes granularity
ticker = 'IJR'
data = yf.download(ticker, period='1mo', interval='30m')

# Instantiate the strategy with the downloaded data and specify the column name for gap calculations
strategy = GapFillOrBounceStrategy(data, close_column_name='Adj Close')

# Identify gaps with specific parameters
gaps = strategy.identify_gaps(gap_percentage=1.0, window_size=1)

# Generate signals
signals = strategy.generate_signals(long_entry=True, short_entry=True)

# Backtest the strategy with an investment amount of $10,000
results = strategy.backtest(investment_amount=10000)

# Display results
print(results)


C:\venv\Investment\investment-311v1\Lib\site-packages\yfinance\scrapers\history.py:239: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  quotes2 = quotes.resample('30T')
[*********************100%%**********************]  1 of 1 completed

                           Open        High         Low       Close  \
Datetime                                                              
2024-07-01 09:30:00  107.010002  107.309998  106.614998  106.885002   
2024-07-01 10:00:00  107.084999  107.129997  105.790001  105.959999   
2024-07-01 10:30:00  105.889999  106.010002  105.654999  105.879997   
2024-07-01 11:00:00  105.790001  105.970001  105.540001  105.580002   
2024-07-01 11:30:00  105.589996  105.709999  105.460098  105.709999   
...                         ...         ...         ...         ...   
2024-07-31 13:30:00  118.809998  118.809998  118.434998  118.650002   
2024-07-31 14:00:00  118.680000  118.680000  118.230003  118.525002   
2024-07-31 14:30:00  118.481003  120.559998  118.481003  120.559998   
2024-07-31 15:00:00  120.559998  120.739998  118.870003  119.065102   
2024-07-31 15:30:00  118.980003  119.180000  118.239998  118.389999   

                      Adj Close  Volume   PrevClose       Gap  GapPercent  \

,Open,Close,LongEntry,LongExit,ShortEntry,ShortExit,Position,PnL,CumulativePnL
Datetime,,,,,,,,,


In [11]:
results.to_csv("file.csv")